# Phase 4: Loss and Training Loop

In [1]:
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
import torch
print(torch.__version__)
print("GPU available:", torch.cuda.is_available())

2.11.0+cu128
GPU available: True


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader

## 1. Combined Loss (weighted cross entropy + dice)

### * Weighted cross entropy

In [4]:
class_weights = torch.tensor([1.0000, 4.6536, 13.0986, 11.8152, 17.2894]) # from phase 2

criterion = nn.CrossEntropyLoss(weight = class_weights)

fake_out = torch.randn(2,5,64,64)
fake_mask = torch.randint(0,5,(2,64,64))

loss = criterion(fake_out, fake_mask)
print("loss:", loss.item())

loss: 1.9680466651916504


### * Dice
true -> 1,1,0,0 : there are 2 buildings
pred -> 1,0,0,0 : model just found 1 of them

In [5]:
true = torch.tensor([1.0, 1.0, 0.0, 0.0])
pred = torch.tensor([1.0, 0.0, 0.0, 0.0])

overlap = (true * pred).sum()
dice = 2 * overlap / (pred.sum() + true.sum())

print("overlap:", overlap.item())
print("dice:", dice.item())
print("dice loss:", 1 - dice.item())

overlap: 1.0
dice: 0.6666666865348816
dice loss: 0.3333333134651184


In [6]:
# softmax: score to probability
probs = torch.softmax(fake_out, dim=1)
print("probs:", probs.shape)
print("sum over 5 classes at one pixel:", probs[0, :, 0, 0].sum().item())

# one hot: 0 or 1
mask_onehot = F.one_hot(fake_mask, num_classes=5).permute(0, 3, 1, 2).float()
print("one-hot:", mask_onehot.shape)
print("one-hot at one pixel:", mask_onehot[0, :, 0, 0])

probs: torch.Size([2, 5, 64, 64])
sum over 5 classes at one pixel: 1.0
one-hot: torch.Size([2, 5, 64, 64])
one-hot at one pixel: tensor([0., 0., 0., 1., 0.])


In [7]:
smooth = 1e-6

overlap = (probs * mask_onehot).sum(dim=(0, 2, 3))
total = probs.sum(dim=(0, 2, 3)) + mask_onehot.sum(dim=(0, 2, 3))

dice_per_class = (2 * overlap + smooth) / (total + smooth)
dice_loss = 1 - dice_per_class.mean()

print("dice per class:", dice_per_class)
print("dice loss:", dice_loss.item())

dice per class: tensor([0.2070, 0.1953, 0.1906, 0.2028, 0.2055])
dice loss: 0.7997745275497437


In [8]:
# weighted cross-entropy on the same fake data
ce_loss = criterion(fake_out, fake_mask)

# combined loss
combined_loss = ce_loss + dice_loss

print("cross-entropy loss:", ce_loss.item())
print("dice loss:", dice_loss.item())
print("combined loss:", combined_loss.item())

cross-entropy loss: 1.9680466651916504
dice loss: 0.7997745275497437
combined loss: 2.7678213119506836


### * Combined Loss

In [24]:
def combined_loss(out, mask, class_weights, smooth=1e-6):
    ce_loss = F.cross_entropy(out, mask, weight=class_weights)

    probs = torch.softmax(out, dim=1)
    mask_onehot = F.one_hot(mask, num_classes=5).permute(0, 3, 1, 2).float()

    overlap = (probs * mask_onehot).sum(dim=(0, 2, 3))
    total = probs.sum(dim=(0, 2, 3)) + mask_onehot.sum(dim=(0, 2, 3))
    dice_per_class = (2 * overlap + smooth) / (total + smooth)

    present = mask_onehot.sum(dim=(0, 2, 3)) > 0
    dice_loss = 1 - dice_per_class[present].mean()

    return ce_loss + dice_loss

In [10]:
# test
loss = combined_loss(fake_out, fake_mask, class_weights)
print("combined loss:", loss.item())

combined loss: 2.7678213119506836


## 2. Training and Validation Loops

In [11]:
import torch
import torch.nn as nn
from torchvision.models import resnet34, ResNet34_Weights

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()

        resnet = resnet34(weights=ResNet34_Weights.IMAGENET1K_V1)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4

    def forward(self, x):
        features = []

        x = self.stem(x)
        features.append(x)

        x1 = self.layer1(self.pool(x))
        features.append(x1)

        x2 = self.layer2(x1)
        features.append(x2)

        x3 = self.layer3(x2)
        features.append(x3)

        x4 = self.layer4(x3)
        features.append(x4)

        return features

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.upsample = nn.Upsample(scale_factor=2)

        self.conv1 = nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU()

    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)

        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        return x

class Decoder(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()

        self.block4 = DecoderBlock(in_channels=1024, skip_channels=512, out_channels=256)
        self.block3 = DecoderBlock(in_channels=256, skip_channels=256, out_channels=128)
        self.block2 = DecoderBlock(in_channels=128, skip_channels=128, out_channels=64)
        self.block1 = DecoderBlock(in_channels=64, skip_channels=128, out_channels=32)

        self.final_upsample = nn.Upsample(scale_factor=2)
        self.final_conv = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, features):
        f0, f1, f2, f3, f4 = features

        x = self.block4(f4, f3)
        x = self.block3(x, f2)
        x = self.block2(x, f1)
        x = self.block1(x, f0)

        x = self.final_upsample(x)
        x = self.final_conv(x)
        return x


class SiameseUNet(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()

        self.encoder = Encoder()
        self.decoder = Decoder(num_classes=num_classes)

    def forward(self, pre, post):
        features_pre = self.encoder(pre)
        features_post = self.encoder(post)

        combined = []
        for f_pre, f_post in zip(features_pre, features_post):
            combined.append(torch.cat([f_pre, f_post], dim=1))

        return self.decoder(combined)

### forward pass

In [12]:
model = SiameseUNet(num_classes=5)

# fake batch
pre = torch.randn(2, 3, 256, 256)
post = torch.randn(2, 3, 256, 256)
mask = torch.randint(0, 5, (2, 256, 256))

# part 1: make a prediction
out = model(pre, post)
print("prediction: ", out.shape)

# part 2: measure how wrong prediction is
loss = combined_loss(out, mask, class_weights)
print("loss:", loss.item())

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 220MB/s]


prediction:  torch.Size([2, 5, 256, 256])
loss: 2.7687153816223145


### backward

In [13]:
# before backward
print("before:", model.decoder.final_conv.weight.grad)

# compute gradients
loss.backward()

# after backward
grad = model.decoder.final_conv.weight.grad
print("after shape:", grad.shape)
print("average size of gradient:", grad.abs().mean().item())

before: None
after shape: torch.Size([5, 32, 1, 1])
average size of gradient: 0.0008761494536884129


### optimizer

In [14]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

w = model.decoder.final_conv.weight
before = w.detach().clone()

optimizer.step()

after = w.detach()
print("weights changed:", not torch.equal(before, after))
print("average change:", (after - before).abs().mean().item())

weights changed: True
average change: 9.999742906074971e-05


### *** training loop test first ***

In [15]:
model = SiameseUNet(num_classes=5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

pre = torch.randn(2, 3, 128, 128)
post = torch.randn(2, 3, 128, 128)
mask = torch.randint(0, 5, (2, 128, 128))

for round_number in range(5):
    optimizer.zero_grad()                           # erase old notes
    out = model(pre, post)                          # prediction
    loss = combined_loss(out, mask, class_weights)  # how wrong?
    loss.backward()                                 # write the notes
    optimizer.step()                                # turn the knobs

    print(f"round {round_number + 1}: loss = {loss.item():.4f}")

round 1: loss = 2.7690
round 2: loss = 2.7670
round 3: loss = 2.7648
round 4: loss = 2.7621
round 5: loss = 2.7591


### Training Loop Function

In [16]:
def train_one_epoch(model, loader, optimizer, class_weights):
    model.train()
    total_loss = 0.0
    
    for pre, post, mask in loader:
        optimizer.zero_grad()
        out = model(pre, post)
        loss = combined_loss(out, mask, class_weights)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss/len(loader)

In [27]:
def validate(model, loader, class_weights, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for pre, post, mask in loader:
            pre = pre.to(device)
            post = post.to(device)
            mask = mask.to(device)

            out = model(pre, post)                          
            loss = combined_loss(out, mask, class_weights)

            total_loss += loss.item()

    return total_loss / len(loader)

In [28]:
val_loss = validate(model, small_loader, class_weights, device)
print("validation loss:", val_loss)

validation loss: 0.9032992422580719


## 3. Overfitting Test On A Small Subset

In [17]:
DATA_ROOT = "/kaggle/input/datasets/azra41221/train-ert/train/train/"
LABELS_DIR = os.path.join(DATA_ROOT, "labels")
TARGETS_DIR = os.path.join(DATA_ROOT, "targets")
IMAGES_DIR = os.path.join(DATA_ROOT, "images")

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class DamageDataset(Dataset):
    def __init__(self, df, images_dir, targets_dir):
        self.base_names = df["base_name"].tolist()
        self.images_dir = images_dir
        self.targets_dir = targets_dir

    def __len__(self):
        return len(self.base_names)

    def _load_image(self, path):
        image = np.array(Image.open(path))
        image = image.astype(np.float32) / 255.0
        image = (image - IMAGENET_MEAN) / IMAGENET_STD
        image = np.transpose(image, (2, 0, 1))
        return image

    def __getitem__(self, idx):
        base_name = self.base_names[idx]

        pre = self._load_image(os.path.join(self.images_dir, f"{base_name}_pre_disaster.png"))
        post = self._load_image(os.path.join(self.images_dir, f"{base_name}_post_disaster.png"))
        mask = np.array(Image.open(os.path.join(self.targets_dir, f"{base_name}_post_disaster_target.png")))

        return torch.from_numpy(pre), torch.from_numpy(post), torch.from_numpy(mask).long()


# small subset for the overfit test
post_label_files = sorted(f for f in os.listdir(LABELS_DIR) if f.endswith("_post_disaster.json"))
small_df = pd.DataFrame({"base_name": [f.replace("_post_disaster.json", "") for f in post_label_files[:8]]})

small_dataset = DamageDataset(small_df, IMAGES_DIR, TARGETS_DIR)
small_loader = DataLoader(small_dataset, batch_size=2, shuffle=True, num_workers=0)

print("samples:", len(small_dataset))
print("batches per epoch:", len(small_loader))

samples: 8
batches per epoch: 4


### GPU

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

class_weights = class_weights.to(device)


def train_one_epoch(model, loader, optimizer, class_weights, device):
    model.train()
    total_loss = 0.0

    for pre, post, mask in loader:
        pre = pre.to(device)
        post = post.to(device)
        mask = mask.to(device)

        optimizer.zero_grad()                           
        out = model(pre, post)                          
        loss = combined_loss(out, mask, class_weights) 
        loss.backward()                                
        optimizer.step()                             

        total_loss += loss.item()

    return total_loss / len(loader)

device: cuda


In [19]:
import time

model = SiameseUNet(num_classes=5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

start = time.time()
avg_loss = train_one_epoch(model, small_loader, optimizer, class_weights, device)
elapsed = time.time() - start

print(f"epoch 1: average loss = {avg_loss:.4f}")
print(f"time: {elapsed:.1f} seconds")
print(f"GPU memory used: {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB")

epoch 1: average loss = 2.8965
time: 14.2 seconds
GPU memory used: 5.1 GB


In [20]:
model = SiameseUNet(num_classes=5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(30):
    avg_loss = train_one_epoch(model, small_loader, optimizer, class_weights, device)
    print(f"epoch {epoch + 1}: average loss = {avg_loss:.4f}")

epoch 1: average loss = 2.8931
epoch 2: average loss = 2.8846
epoch 3: average loss = 2.8771
epoch 4: average loss = 2.8715
epoch 5: average loss = 2.8655
epoch 6: average loss = 2.8594
epoch 7: average loss = 2.8558
epoch 8: average loss = 2.8519
epoch 9: average loss = 2.8490
epoch 10: average loss = 2.8464
epoch 11: average loss = 2.8445
epoch 12: average loss = 2.8392
epoch 13: average loss = 2.8384
epoch 14: average loss = 2.8388
epoch 15: average loss = 2.8356
epoch 16: average loss = 2.8333
epoch 17: average loss = 2.8324
epoch 18: average loss = 2.8337
epoch 19: average loss = 2.8320
epoch 20: average loss = 2.8337
epoch 21: average loss = 2.8311
epoch 22: average loss = 2.8289
epoch 23: average loss = 2.8279
epoch 24: average loss = 2.8298
epoch 25: average loss = 2.8292
epoch 26: average loss = 2.8261
epoch 27: average loss = 2.8280
epoch 28: average loss = 2.8282
epoch 29: average loss = 2.8244
epoch 30: average loss = 2.8269


In [21]:
model = SiameseUNet(num_classes=5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

pre, post, mask = next(iter(small_loader))
pre, post, mask = pre.to(device), post.to(device), mask.to(device)

model.train()
for step in range(40):
    optimizer.zero_grad()
    out = model(pre, post)
    loss = combined_loss(out, mask, class_weights)
    loss.backward()
    optimizer.step()

    if step % 5 == 0:
        print(f"step {step}: loss = {loss.item():.4f}")

step 0: loss = 2.8859
step 5: loss = 2.8214
step 10: loss = 2.7936
step 15: loss = 2.7772
step 20: loss = 2.7643
step 25: loss = 2.7519
step 30: loss = 2.7403
step 35: loss = 2.7295


In [22]:
model.train()
for step in range(40, 160):
    optimizer.zero_grad()
    out = model(pre, post)
    loss = combined_loss(out, mask, class_weights)
    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        print(f"step {step}: loss = {loss.item():.4f}")

step 40: loss = 2.7198
step 50: loss = 2.6992
step 60: loss = 2.6804
step 70: loss = 2.6644
step 80: loss = 2.6491
step 90: loss = 2.6361
step 100: loss = 2.6272
step 110: loss = 2.6158
step 120: loss = 2.6075
step 130: loss = 2.6012
step 140: loss = 2.5964
step 150: loss = 2.5927


In [23]:
with torch.no_grad():
    ce = F.cross_entropy(out, mask, weight=class_weights)

    probs = torch.softmax(out, dim=1)
    mask_onehot = F.one_hot(mask, num_classes=5).permute(0, 3, 1, 2).float()
    overlap = (probs * mask_onehot).sum(dim=(0, 2, 3))
    total = probs.sum(dim=(0, 2, 3)) + mask_onehot.sum(dim=(0, 2, 3))
    dice_per_class = (2 * overlap + 1e-6) / (total + 1e-6)

print("cross-entropy part:", ce.item())
print("dice loss part:", (1 - dice_per_class.mean()).item())
print("dice per class:", dice_per_class)
print("pixels per class in this batch:", torch.bincount(mask.flatten(), minlength=5))

cross-entropy part: 0.0614011287689209
dice loss part: 0.6220380067825317
dice per class: tensor([9.7189e-01, 9.1792e-01, 2.6697e-11, 2.9503e-11, 3.4913e-11],
       device='cuda:0')
pixels per class in this batch: tensor([2030115,   67037,       0,       0,       0], device='cuda:0')


In [26]:
model = SiameseUNet(num_classes=5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(20):
    avg_loss = train_one_epoch(model, small_loader, optimizer, class_weights, device)
    print(f"epoch {epoch + 1}: average loss = {avg_loss:.4f}")

epoch 1: average loss = 2.3292
epoch 2: average loss = 2.0799
epoch 3: average loss = 1.8409
epoch 4: average loss = 1.7394
epoch 5: average loss = 1.6278
epoch 6: average loss = 1.5409
epoch 7: average loss = 1.4720
epoch 8: average loss = 1.4080
epoch 9: average loss = 1.3334
epoch 10: average loss = 1.2385
epoch 11: average loss = 1.2083
epoch 12: average loss = 1.1712
epoch 13: average loss = 1.1669
epoch 14: average loss = 1.0888
epoch 15: average loss = 0.9929
epoch 16: average loss = 1.0165
epoch 17: average loss = 0.9926
epoch 18: average loss = 0.9542
epoch 19: average loss = 0.9021
epoch 20: average loss = 0.8524
